# Hyperparameter Fine-Tuning with CV: Random Forest & MLP
Uses `RandomizedSearchCV` with `AugmentingClassifier` so augmentation happens inside each fold — no leakage.

In [1]:
from pathlib import Path
import os
import numpy as np
import cv2
from tqdm.auto import tqdm

IMAGES_PATH = Path("../resources/cloud-images/cloud-classifier-1")

def index_labeled_images(images_path=IMAGES_PATH):
    cloud_labels = []
    images_path = Path(images_path)
    labeled_images = {}
    if not images_path.exists():
        return labeled_images, cloud_labels
    for cloud_dir in sorted(p for p in images_path.iterdir() if p.is_dir()):
        cloud_labels.append(cloud_dir.name)
        for img_path in sorted(p for p in cloud_dir.iterdir() if p.is_file()):
            labeled_images[img_path.name] = {"label": cloud_dir.name, "path": str(img_path)}
    return labeled_images, cloud_labels

def extract_labels(labeled_images):
    images, labels = [], []
    for image_name in tqdm(labeled_images, total=len(labeled_images)):
        path = labeled_images[image_name]['path']
        if not os.path.exists(path):
            continue
        image = cv2.imread(path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.resize(image, (128, 128))
        images.append(image)
        labels.append(labeled_images[image_name]['label'])
    return np.array(images), np.array(labels)

labeled_images, cloud_labels = index_labeled_images()
images, labels = extract_labels(labeled_images)
print(f"Loaded {len(images)} images, classes: {cloud_labels}")

/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 698/698 [00:00<00:00, 1205.28it/s]

Loaded 698 images, classes: ['clear_sky', 'cloud']


In [2]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder

ordinal_encoder = LabelEncoder()
labels_encoded = ordinal_encoder.fit_transform(labels)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(images, labels_encoded))

X_train, X_test = images[train_idx], images[test_idx]
y_train, y_test = labels_encoded[train_idx], labels_encoded[test_idx]

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)

train: (558, 128, 128) (558,)
test:  (140, 128, 128) (140,)


In [3]:
import torch
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (128, 128)
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0, translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR, fill=0
)
wrap_translation = T.Lambda(lambda x: torch.roll(
    x,
    shifts=(
        int(torch.randint(-int(MAX_FRAC * x.shape[-2]), int(MAX_FRAC * x.shape[-2]) + 1, (1,)).item()),
        int(torch.randint(-int(MAX_FRAC * x.shape[-1]), int(MAX_FRAC * x.shape[-1]) + 1, (1,)).item()),
    ),
    dims=(-2, -1),
))

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(degrees=20, scale=(0.90, 1.10), interpolation=InterpolationMode.BILINEAR, fill=0),
    T.RandomResizedCrop(size=IMG_SIZE, scale=(0.80, 1.00), ratio=(0.90, 1.10), interpolation=InterpolationMode.BILINEAR),
])
one_of = T.RandomChoice([
    wrap_translation, border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

augmentation_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
])

In [4]:
from sklearn.base import BaseEstimator, ClassifierMixin

class AugmentingClassifier(BaseEstimator, ClassifierMixin):
    """
    Wraps any sklearn classifier so augmentation happens inside fit() only.
    Validation folds always see unaugmented originals — no leakage.
    """
    def __init__(self, base_clf=None, target_count=2400, random_state=42):
        self.base_clf = base_clf
        self.target_count = target_count
        self.random_state = random_state

    def _augment(self, X, y):
        X_extra, y_extra = [], []
        for cls in np.unique(y):
            cls_imgs = X[y == cls]
            multiple = self.target_count // len(cls_imgs)
            for img in cls_imgs:
                img_torch = torch.from_numpy(img)
                for _ in range(multiple - 1):
                    aug = augmentation_transform(img_torch.unsqueeze(0)).squeeze(0).numpy()
                    X_extra.append(aug)
                    y_extra.append(cls)

        X_norm = X.astype(np.float32) / 255.0
        if X_extra:
            X_all = np.concatenate([X_norm, np.stack(X_extra)])
            y_all = np.concatenate([y, np.array(y_extra)])
        else:
            X_all, y_all = X_norm, y

        rng = np.random.RandomState(self.random_state)
        idx = rng.permutation(len(X_all))
        return X_all[idx], y_all[idx]

    def fit(self, X, y):
        X_aug, y_aug = self._augment(X, y)
        self.base_clf.fit(X_aug.reshape(len(X_aug), -1), y_aug)
        return self

    def predict(self, X):
        X_norm = X.astype(np.float32) / 255.0
        return self.base_clf.predict(X_norm.reshape(len(X_norm), -1))

    def predict_proba(self, X):
        X_norm = X.astype(np.float32) / 255.0
        return self.base_clf.predict_proba(X_norm.reshape(len(X_norm), -1))

## Random Forest — RandomizedSearchCV

In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint

rf_param_dist = {
    "base_clf__n_estimators":          randint(50, 150),
    "base_clf__max_depth":             [5, 10, 15, 20],
    "base_clf__min_samples_split":     randint(2, 15),
    "base_clf__min_samples_leaf":      randint(1, 8),
    "base_clf__max_features":          ["sqrt", "log2"],
    "base_clf__criterion":             ["gini", "entropy"],
    "base_clf__bootstrap":             [True, False],
    "base_clf__min_impurity_decrease": [0.0, 0.01, 0.05],
}

rf_search = RandomizedSearchCV(
    AugmentingClassifier(base_clf=RandomForestClassifier(random_state=42, n_jobs=-1)),
    rf_param_dist,
    n_iter=30,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1,
    random_state=42,
    verbose=1,
    refit=True
)

rf_search.fit(X_train, y_train)

print(f"\nBest params: {rf_search.best_params_}")
print(f"Best CV F1:  {rf_search.best_score_:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

y_pred = rf_search.best_estimator_.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro:  {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=ordinal_encoder.classes_))

## MLP — RandomizedSearchCV

In [ ]:
# MLP fine-tuning — to be filled in